# Import Modules

In [91]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_percentage_error
from hijridate import Hijri, Gregorian
from statsmodels.graphics.tsaplots import plot_pacf
from statsmodels.tsa.stattools import pacf

# Data Selection

In [92]:
raw_datasets: dict[str, pd.DataFrame] = {}

# Specify where datasets stored
raw_dataset_dir = Path("../dataset/raw")
paths = sorted(raw_dataset_dir.glob("*.xlsx"))

# Check if datasets exists
if not paths:
    raise FileNotFoundError(
        f"Tidak ditemukan dataset .xlsx pada {raw_dataset_dir.resolve()}"
    )

# Print loaded dataset
print(f"Loaded Dataset :")
for path in paths:
    name = path.stem
    print(f"{path.name}")
    df_raw = pd.read_excel(path)
    raw_datasets[name] = df_raw

Loaded Dataset :
2018-01-01_2018-12-31.xlsx
2019-01-01_2019-12-31.xlsx
2020-01-01_2020-12-31.xlsx
2021-01-01_2021-12-31.xlsx
2022-01-03_2022-12-30.xlsx
2023-01-02_2023-12-29.xlsx
2024-01-01_2024-12-31.xlsx
2025-01-01_2025-11-28.xlsx


# Data Cleaning

In [93]:
clean_dataset_dir = Path("../dataset/clean")
clean_dataset_dir.mkdir(parents=True, exist_ok=True)

cleaned_datasets: dict[str, pd.DataFrame] = {}

# Loop process to clean each raw datasets
for name, df in raw_datasets.items():
    all_cols = df.columns.tolist()
    num_col = all_cols[0]
    province_col = all_cols[1]

    # Save original row order
    df["prov_order"] = range(len(df))

    # Rename column to province
    df = df.rename(columns={province_col: "Province"})

    # Clean date column names
    date_cols = [c for c in df.columns if c not in [num_col, "Province"]]

    # Remove space gap on date
    rename_map = {c: c.replace(" ", "") for c in date_cols}
    df = df.rename(columns=rename_map)

    # Recompute date_cols after rename
    date_cols = [c for c in df.columns if c not in [num_col, "Province", "prov_order"]]

    # Melt wide to long format
    df_long = df.melt(
        id_vars=["prov_order", "Province"],
        value_vars=date_cols,
        var_name="Date",
        value_name="Price",
    )

    # Format date to datetime
    df_long["Date"] = pd.to_datetime(df_long["Date"], format="%d/%m/%Y")

    # Replace "-" with NaN & remove comma from all price and convert it to numeric
    df_long["Price"] = (
        df_long["Price"]
        .astype(str)
        .str.strip()
        .replace("-", np.nan)
        .str.replace(",", "", regex=False)
    )
    df_long["Price"] = pd.to_numeric(df_long["Price"], errors="coerce")

    # Sort data
    df_long = df_long.sort_values(["Date", "prov_order"]).reset_index(drop=True)
    df_long = df_long.drop(columns=["prov_order"])
    df_long = df_long[["Date", "Price", "Province"]]

    # Interpolate by province
    df_long["Price"] = df_long.groupby("Province", group_keys=False)["Price"].apply(
        lambda s: s.interpolate(method="linear").ffill().bfill()
    )

    # Fill missing date
    df_long = (
        df_long.groupby("Province", group_keys=False)[
            ["Date", "Price", "Province"]
        ]
        .apply(
            lambda g: (
                g.set_index("Date")
                .reindex(
                    pd.date_range(
                        start=g["Date"].min(),
                        end=g["Date"].max(),
                        freq="D",
                    )
                )
                .assign(Province=lambda x: x["Province"].ffill().bfill())
                .assign(
                    Price=lambda x: x["Price"].interpolate("linear").ffill().bfill()
                )
                .rename_axis("Date")
                .reset_index()
            )
        )
        .reset_index(drop=True)
    )

    # Round price
    df_long["Price"] = df_long["Price"].round().astype(int)
    
    # Export cleaned dataframe
    file_name = f"Clean_{name}.xlsx"
    df_long.to_excel(clean_dataset_dir / file_name)

    # Print preview cleaned dataframe
    print("\n")
    print(f"Dataset {name}")
    print(df_long.head(5))
    print("...")
    print(df_long.tail(5))
    print("\n")
    
    cleaned_datasets[name] = df_long



Dataset 2018-01-01_2018-12-31
        Date  Price Province
0 2018-01-01  14600     Aceh
1 2018-01-02  14600     Aceh
2 2018-01-03  14550     Aceh
3 2018-01-04  14500     Aceh
4 2018-01-05  14500     Aceh
...
           Date  Price        Province
8390 2018-12-27  13850  Sumatera Barat
8391 2018-12-28  13850  Sumatera Barat
8392 2018-12-29  13850  Sumatera Barat
8393 2018-12-30  13850  Sumatera Barat
8394 2018-12-31  13850  Sumatera Barat




Dataset 2019-01-01_2019-12-31
        Date  Price Province
0 2019-01-01  14250     Aceh
1 2019-01-02  14250     Aceh
2 2019-01-03  14250     Aceh
3 2019-01-04  14250     Aceh
4 2019-01-05  14250     Aceh
...
           Date  Price        Province
8390 2019-12-27  14300  Sumatera Barat
8391 2019-12-28  14317  Sumatera Barat
8392 2019-12-29  14333  Sumatera Barat
8393 2019-12-30  14350  Sumatera Barat
8394 2019-12-31  14350  Sumatera Barat




Dataset 2020-01-01_2020-12-31
        Date  Price Province
0 2020-01-01  14500     Aceh
1 2020-01-02  1450

# Data Integration

In [94]:
merged_dataset_path = Path("../dataset/merged/merged_datasets.xlsx")
merged_dataset_path.parent.mkdir(parents=True, exist_ok=True)

if not cleaned_datasets:
    raise ValueError("Datasets kosong, pastikan Data Cleaning sudah dijalankan.")

frames: list[pd.DataFrame] = []

for name, df_transform in cleaned_datasets.items():
    new_df = df_transform.copy()
    frames.append(new_df)

# Gabung semua dataset
df_merged = pd.concat(frames, ignore_index=True)

# Normalisasi tipe data
df_merged["Date"] = pd.to_datetime(df_merged["Date"])
df_merged["Province"] = df_merged["Province"].astype("string")
df_merged["Price"] = pd.to_numeric(df_merged["Price"], errors="coerce")

# Urutkan dan hilangkan duplikat per (Province, Date)
# kalau ada lebih dari satu sumber di tanggal yg sama, ambil yang terakhir
df_merged = df_merged.drop_duplicates(
    subset=["Province", "Date"], keep="last"
).reset_index(drop=True)

# Export merged dataframe
df_merged.to_excel(merged_dataset_path)

# Print preview merged dataframe
print("\n")
print(df_merged.head(5))
print("...")
print(df_merged.tail(5))
print("\n")



        Date  Price Province
0 2018-01-01  14600     Aceh
1 2018-01-02  14600     Aceh
2 2018-01-03  14550     Aceh
3 2018-01-04  14500     Aceh
4 2018-01-05  14500     Aceh
...
            Date  Price        Province
66304 2025-11-24  18250  Sumatera Barat
66305 2025-11-25  18250  Sumatera Barat
66306 2025-11-26  18250  Sumatera Barat
66307 2025-11-27  18250  Sumatera Barat
66308 2025-11-28  18250  Sumatera Barat




# Data Transformation

In [95]:
def eid_delta_days(ts: pd.Timestamp) -> int:
    g = Gregorian(ts.year, ts.month, ts.day)
    h = g.to_hijri()

    # 1 Syawal = 10 Hijriah
    eid_h = Hijri(h.year, 10, 1)
    eid_g = eid_h.to_gregorian()

    eid_date = pd.Timestamp(eid_g.year, eid_g.month, eid_g.day)
    return (ts.normalize() - eid_date).days

def eid_flags(ts: pd.Timestamp):
    d = eid_delta_days(ts)
    before = 1 if -7 <= d <= -1 else 0
    day     = 1 if d == 0 else 0
    after  = 1 if 1 <= d <= 6 else 0
    return pd.Series([before, day, after],
                     index=["before_eid", "eid", "after_eid"])

transformed_dataset_path = Path("../dataset/transformed/transformed_datasets.xlsx")
transformed_dataset_path.parent.mkdir(parents=True, exist_ok=True)

df = df_merged.copy()

# Pastikan tipe data rapi
df["Date"] = pd.to_datetime(df["Date"])
df["Province"] = df["Province"].astype("string")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

# 1) Province → numeric (ID)
df = df.sort_values(["Province", "Date"]).reset_index(drop=True)
df["Province_id"] = df["Province"].astype("category").cat.codes

# 2) Lag features per provinsi
for lag in [1, 2, 3, 13]:
    df[f"lag_{lag}"] = df.groupby("Province", group_keys=False)["Price"].shift(lag)

# 3) Fitur Moving Holiday
df[["before_eid", "eid", "after_eid"]] = df["Date"].apply(eid_flags)

# 4) Time feature
df["month"] = df["Date"].dt.month
df["year"] = df["Date"].dt.year

# Buang baris yang belum punya semua lag (awal-awal tiap provinsi)
df_transform = df.dropna(subset=["lag_1", "lag_2", "lag_3", "lag_13"]).reset_index(drop=True)

# Export transformed dataframe
df_transform.to_excel(transformed_dataset_path)

# Print preview transformed dataframe
print("\n")
print(df_transform.head(5))
print("...")
print(df_transform.tail(5))
print("\n")



        Date  Price Province  Province_id    lag_1    lag_2    lag_3   lag_13  \
0 2018-01-14  14650     Aceh            0  14650.0  14650.0  14650.0  14600.0   
1 2018-01-15  14650     Aceh            0  14650.0  14650.0  14650.0  14600.0   
2 2018-01-16  14650     Aceh            0  14650.0  14650.0  14650.0  14550.0   
3 2018-01-17  14650     Aceh            0  14650.0  14650.0  14650.0  14500.0   
4 2018-01-18  14600     Aceh            0  14650.0  14650.0  14650.0  14500.0   

   before_eid  eid  after_eid  month  year  
0           0    0          0      1  2018  
1           0    0          0      1  2018  
2           0    0          0      1  2018  
3           0    0          0      1  2018  
4           0    0          0      1  2018  
...
            Date  Price        Province  Province_id    lag_1    lag_2  \
66005 2025-11-24  18250  Sumatera Barat           22  18250.0  18250.0   
66006 2025-11-25  18250  Sumatera Barat           22  18250.0  18250.0   
66007 2025-11-2

# Data Mining

In [96]:
# Configure Parameters, Train Size, etc
mined_dataset_path = Path("../dataset/mined/rfr_datasets.xlsx")
mined_dataset_path.parent.mkdir(parents=True, exist_ok=True)
df = df_transform.copy()

RFR_PARAMS = dict(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    bootstrap=True,
    random_state=42,
    n_jobs=-1,
)

train_size = 0.8

In [97]:
# Random Forest Regression Process
df["Date"] = pd.to_datetime(df["Date"])

# Sort by time to keep chronological order for split
df = df.sort_values("Date").reset_index(drop=True)

feature_cols = ["Province_id", "lag_1", "lag_2", "lag_3", "lag_13", "before_eid", "eid", "after_eid"]
target_col = "Price"

X = df[feature_cols]
y = df[target_col]

# Time-based train-test split
split_index = int(len(df) * train_size)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

model = RandomForestRegressor(**RFR_PARAMS)

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Model Evaluation

In [98]:
rmse = root_mean_squared_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100

# Knowledge Presentation

In [99]:
print("\n")
print("=== Random Forest Regression (Data Mining) ===")
print(f"Train Data : {train_size*100}%")
print(f"RMSE : {rmse:,.2f}")
print(f"MAPE : {mape:.2f}%")

# Save prediction results for analysis
mined_df = pd.DataFrame(
    {
        "Date": df.iloc[split_index:]["Date"].values,
        "Province": df.iloc[split_index:]["Province"].values,
        "Actual": y_test.values,
        "Prediction": y_pred,
    }
)

mined_df.to_excel(mined_dataset_path, index=False)



=== Random Forest Regression (Data Mining) ===
Train Data : 80.0%
RMSE : 495.36
MAPE : 0.65%


# Optional

In [ ]:
# Sugar Price History Plot
df = df_merged.copy()

# ========= OUTPUT PATHS =========
combined_path = Path("../output/plots/sugar_price_history.png")
province_dir = Path("../output/plots/province_price_history/")
combined_path.parent.mkdir(parents=True, exist_ok=True)
province_dir.mkdir(parents=True, exist_ok=True)

# ========= COMBINED PLOT =========
plt.figure(figsize=(12, 6))

for prov, g in df.groupby("Province"):
    plt.plot(g["Date"], g["Price"], label=prov, linewidth=0.8, alpha=0.7)

plt.title("Riwayat Harga Gula per Provinsi")
plt.xlabel("Tanggal")
plt.ylabel("Harga (Rp)")
plt.legend(
    title="Provinsi", fontsize=6, ncol=2, bbox_to_anchor=(1.05, 1), loc="upper left"
)
plt.tight_layout()
plt.savefig(combined_path)
plt.close()

# ========= INDIVIDUAL PROVINCE PLOTS =========
for prov, g in df.groupby("Province"):
    plt.figure(figsize=(10, 5))
    plt.plot(g["Date"], g["Price"], linewidth=1.5)

    plt.title(f"Riwayat Harga Gula - {prov}")
    plt.xlabel("Tanggal")
    plt.ylabel("Harga (Rp)")
    plt.tight_layout()

    # clean filename
    file_name = prov.replace(" ", "_").replace("/", "_") + ".png"
    plt.savefig(province_dir / file_name)
    plt.close()

In [ ]:
# Generate PACF Plot
df = df_transform.copy()

y_full = df['Price'].astype(float)

max_lag = 31

# 1) Plot PACF
fig, ax = plt.subplots(figsize=(10, 4))
plot_pacf(y_full, ax=ax, lags=max_lag, method="ywm")  # method Yule-Walker Modified
ax.set_title(f"PACF Harga Gula Pasir (1 s.d. {max_lag} lag)")
ax.set_xlabel("Lag")
ax.set_ylabel("Partial Autocorrelation")
plt.tight_layout()
plt.show()

# 2) Ambil nilai PACF secara numerik + interval kepercayaan
pacf_vals, confint = pacf(
    y_full,
    nlags=max_lag,
    alpha=0.05,   # 95% confidence interval
    method="ywm"
)

# 3) Cari lag yang signifikan (garis batang keluar dari band interval nol)
significant_lags = [
    lag for lag in range(1, len(pacf_vals))
    if (confint[lag, 0] > 0) or (confint[lag, 1] < 0)
]

print("Lag yang signifikan menurut PACF:", significant_lags)

In [100]:
# Compare Year Prediction
target_year = 2025

output_dir = Path("../output/xlsx")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"rfr_year_comparison_{target_year}.xlsx"
compare_plot_dir = Path("../output/plots/compare")
compare_plot_dir.mkdir(parents=True, exist_ok=True)

df = df_transform.copy()

feature_cols = ["Province_id", "lag_1", "lag_2", "lag_3", "lag_13", "before_eid", "eid", "after_eid"]
target_col = "Price"

# Split train / test by year
train_mask = df["Date"].dt.year < target_year
test_mask = df["Date"].dt.year == target_year

if not train_mask.any():
    raise ValueError(f"Tidak ada data sebelum tahun {target_year} untuk training.")
if not test_mask.any():
    raise ValueError(f"Tidak ada data pada tahun {target_year} untuk pengujian.")

df_train = df[train_mask].sort_values("Date")
df_test = df[test_mask].sort_values("Date")

X_train = df_train[feature_cols]
y_train = df_train[target_col]

X_test = df_test[feature_cols]
y_test = df_test[target_col]

model = RandomForestRegressor(**RFR_PARAMS)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Evaluasi keseluruhan tahun
rmse = root_mean_squared_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100

print(f"\n=== Year-wise Comparison for {target_year} ===")
print(f"Train data up to: {df_train['Date'].max().date()}")
print(f"RMSE : {rmse:,.2f}")
print(f"MAPE : {mape:.2f}%")

# Simpan hasil ke Excel
result_df = pd.DataFrame(
    {
        "Date": df_test["Date"].values,
        "Province": df_test["Province"].values,
        "Actual": y_test.values,
        "Prediction": y_pred,
    }
)

result_df.to_excel(output_path, index=False)

# ========== HISTOGRAM PER PROVINSI ==========
for prov in sorted(result_df["Province"].unique()):
    sub = result_df[result_df["Province"] == prov][["Date", "Actual", "Prediction"]]

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(sub["Date"], sub["Actual"], label="Actual", linewidth=1.2)
    ax.plot(sub["Date"], sub["Prediction"], label="Prediction", linewidth=1.2)

    ax.set_title(f"Riwayat Harga Aktual vs Prediksi - {prov} ({target_year})")
    ax.set_xlabel("Tanggal")
    ax.set_ylabel("Harga (Rp)")
    ax.legend()

    fig.tight_layout()
    fig.savefig(compare_plot_dir / f"{prov}_{target_year}.png")

    plt.close(fig)


=== Year-wise Comparison for 2025 ===
Train data up to: 2024-12-31
RMSE : 526.72
MAPE : 0.57%


In [ ]:
df = df_transform.copy()

feature_cols = [
    "Province_id", 
    "lag_1", "lag_2", "lag_3", "lag_13",
    "before_eid", "eid", "after_eid"
]
target_col = "Price"

X = df[feature_cols]
y = df[target_col]

model = RandomForestRegressor(**RFR_PARAMS)
model.fit(X_train, y_train)

horizon = 365
forecast_results = []

provinces = df["Province"].unique()

forecast_plot_dir = Path("../output/plots/forecast/")
forecast_plot_dir.mkdir(parents=True, exist_ok=True)

# Forecast loop each province
for prov in provinces:
    # ---- ambil histori provinsi ----
    g = df[df["Province"] == prov].copy()
    g = g.sort_values("Date").reset_index(drop=True)

    last_date = g["Date"].max()
    last_rows = g.tail(30).copy()

    future_rows = []

    prov_id = g.iloc[-1]["Province_id"]

    # Forecast loop until specified horizon
    for i in range(1, horizon + 1):

        next_date = last_date + pd.Timedelta(days=i)

        # Lag Features
        lag_1  = last_rows.iloc[-1]["Price"]
        lag_2  = last_rows.iloc[-2]["Price"]  if len(last_rows) >= 2  else lag_1
        lag_3 = last_rows.iloc[-3]["Price"] if len(last_rows) >= 3 else lag_1
        lag_13 = last_rows.iloc[-13]["Price"] if len(last_rows) >= 13 else lag_1

        # Eid Features
        gdate = Gregorian(next_date.year, next_date.month, next_date.day)
        h = gdate.to_hijri()

        # 1 Syawal (Idul Fitri)
        eid_h = Hijri(h.year, 10, 1)
        eid_g = eid_h.to_gregorian()
        eid_date = pd.Timestamp(eid_g.year, eid_g.month, eid_g.day)

        delta = (next_date.normalize() - eid_date).days

        before_eid = 1 if -7 <= delta <= -1 else 0
        eid        = 1 if delta == 0 else 0
        after_eid  = 1 if 1 <= delta <= 6 else 0

        # Arrange future data features
        X_future = pd.DataFrame({
            "Province_id": [prov_id],
            "lag_1": [lag_1],
            "lag_2": [lag_2],
            "lag_3": [lag_3],
            "lag_13": [lag_13],
            "before_eid": [before_eid],
            "eid": [eid],
            "after_eid": [after_eid],
        })

        predicted_price = model.predict(X_future)[0]

        forecast_results.append({
            "Date": next_date,
            "Province": prov,
            "Prediction": round(predicted_price),
        })

        future_rows.append({
            "Date": next_date,
            "Price": predicted_price
        })

        # Update series
        last_rows = pd.concat([
            last_rows,
            pd.DataFrame([{
                "Date": next_date,
                "Price": predicted_price,
                "Province": prov,
                "Province_id": prov_id,
            }])
        ], ignore_index=True)


    # Plot
    future_df = pd.DataFrame(future_rows)

    plt.figure(figsize=(12, 5))
    plt.plot(g["Date"], g["Price"], label="Historical", linewidth=1.5)
    plt.plot(future_df["Date"], future_df["Price"], label="Forecast (1 Year)", linewidth=1.5)

    plt.title(f"Historical + 1-Year Forecast Harga Gula - {prov}")
    plt.xlabel("Tanggal")
    plt.ylabel("Harga (Rp)")
    plt.legend()
    plt.tight_layout()

    file_name = prov.replace(" ", "_").replace("/", "_") + "_forecast.png"
    plt.savefig(forecast_plot_dir / file_name, dpi=150, bbox_inches="tight")
    plt.close()

df_forecast = pd.DataFrame(forecast_results)
df_forecast = df_forecast.sort_values(["Province", "Date"]).reset_index(drop=True)